<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms-W2/blob/main/W2D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import requests
import sqlite3
import time
from datetime import datetime

# API settings
API_KEY = "YOUR_API_KEY"
CITY = "Alexandria"
API_URL = "https://api.openweathermap.org/data/2.5/weather"

# Webhook URL from yesterday's task
WEBHOOK_URL = "YOUR_WEBHOOK_URL/webhook"

# Database setup
def setup_database():
    connection = sqlite3.connect("store.db")

    connection.execute("""
        CREATE TABLE IF NOT EXISTS weather_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            city TEXT,
            temperature REAL,
            description TEXT,
            created_at TEXT
        )
    """)

    connection.commit()
    connection.close()


def run_pipeline():
    # Step 1: Get data from the API
    response = requests.get(
        API_URL,
        params={
            "q": CITY,
            "appid": API_KEY,
            "units": "metric"
        },
        timeout=15
    )

    if response.status_code != 200:
        print("API request failed:", response.status_code)
        return False

    data = response.json()

    # Extract the required data
    city = data["name"]
    temperature = data["main"]["temp"]
    description = data["weather"][0]["description"]
    created_at = datetime.now().isoformat(timespec="seconds")

    print("API request successful!")

    # Step 2: Store the data in SQLite
    connection = sqlite3.connect("store.db")

    connection.execute("""
        INSERT INTO weather_data
        (city, temperature, description, created_at)
        VALUES (?, ?, ?, ?)
    """, (city, temperature, description, created_at))

    connection.commit()
    connection.close()

    print("Data saved to store.db")

    # Step 3: Send the same data to the webhook
    payload = {
        "city": city,
        "temperature": temperature,
        "description": description,
        "created_at": created_at
    }

    webhook_response = requests.post(
        WEBHOOK_URL,
        json=payload,
        timeout=15
    )

    if webhook_response.status_code not in range(200, 300):
        print("Webhook request failed:", webhook_response.status_code)
        return False

    print("Data sent to webhook successfully!")
    return True


if __name__ == "__main__":
    setup_database()

    cycle = 1

    while True:
        print(f"\nStarting pipeline cycle {cycle}...")

        try:
            success = run_pipeline()

            if success:
                print(f"تم تنفيذ الدورة رقم {cycle} بنجاح")
            else:
                print(f"الدورة رقم {cycle} لم تكتمل")

        except (requests.RequestException, KeyError, ValueError, sqlite3.Error) as error:
            print(f"حدث خطأ في الدورة رقم {cycle}: {error}")

        cycle += 1
        print("Waiting 30 seconds...")
        time.sleep(30)